# 01 · 群体结构与基因型 PCA Population Structure & Genotype PCA

> **能力标签**：GB3（群体结构与混合模型 / Population Structure & Mixed Models）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**群体结构（population structure / population stratification）**如何成为 GWAS 的混杂因子（confounder），造成虚假关联（spurious association）。
2. 理解并实现**基因型主成分分析（genotype PCA）**：先标准化基因型矩阵，再用 PCA 提取主成分（principal components, PC）。
3. 把 **PC 作为协变量**加入回归模型，校正群体分层带来的统计膨胀。
4. 用**基因组膨胀因子** $\lambda_{\text{GC}}$ 量化校正效果——加入 PC 后 $\lambda_{\text{GC}}$ 应接近 1。
5. 为 **b8-genopca** 作业做好准备：实现 `genotype_pca(G, k)` 函数。

> 对应能力：**GB3**


## 概念讲解 Concepts

### 1 · 群体结构作为混杂因子 Population Structure as Confounder

在多民族（multi-ethnic）或地理来源混合的样本中，不同个体之间存在**系统性的遗传差异**，称为**群体分层（population stratification）**。

若研究的表型（如身高、BMI）在不同群体中存在均值差异，同时这些群体在全基因组数以千计的 SNP 上都有等位基因频率差异，
那么即使某个 SNP 与表型毫无因果关系，仅因为它的频率在两群体间不同，也会产生统计关联——即**虚假关联（spurious association）**。

**结果**：QQ 图整体上移，$\lambda_{\text{GC}} \gg 1$（系统性膨胀），误报率（false positive rate）急剧升高。

---

### 2 · 基因型 PCA Genotype PCA

**思路**：把 $n \times m$ 基因型矩阵 $G$（$n$ 样本，$m$ SNP）做主成分分析（PCA），
前几个主成分（PC1, PC2, …）能捕捉最大的遗传变异方向——通常对应不同的祖先来源（ancestry）。

**标准化步骤**（每列 SNP）：

$$G^*_{ij} = \frac{G_{ij} - \bar{G}_j}{\text{SD}(G_j) + \epsilon}$$

其中 $\epsilon = 10^{-8}$ 防止除零（单态 SNP，monomorphic sites）。

**PCA**：对 $G^* \in \mathbb{R}^{n \times m}$ 做截断 SVD（truncated SVD），前 $k$ 个左奇异向量（left singular vectors）
即为 $k$ 个主成分得分矩阵 $P \in \mathbb{R}^{n \times k}$。

sklearn 的 `PCA(n_components=k).fit_transform(G*)` 直接实现以上步骤。

---

### 3 · PC 作为协变量 PCs as Covariates

将 PCA 得到的前 $k$ 个 PC 作为协变量加入 GWAS 线性回归：

$$y = \mu + \beta_j g_j + \gamma_1 \text{PC}_1 + \cdots + \gamma_k \text{PC}_k + \varepsilon$$

借助 FWL 定理（Frisch-Waugh-Lovell theorem），这等价于先把 $y$ 和所有基因型 $g_j$ 对 PC 做残差化，
再进行简单回归。PC 的列空间（column space）代表群体分层信息，"投影出去"后群体效应被移除。

**常用设置**：GWAS 中通常加入前 10–20 个 PC，具体数量由 scree plot（碎石图）或协议决定。

---

### 4 · $\lambda_{\text{GC}}$ 的改善

**基因组膨胀因子（genomic inflation factor）**：

$$\lambda_{\text{GC}} = \frac{\text{median}(\chi^2_{\text{obs}})}{\text{median}(\chi^2_{1, 0.5})} \approx \frac{\text{median}(t^2)}{0.4549}$$

- 不加 PC：群体分层导致 $\chi^2$ 统计量系统性偏大，$\lambda_{\text{GC}} > 1$
- 加入 PC 后：$\lambda_{\text{GC}} \rightarrow 1.0$（理想情况），说明系统性偏差被校正

> **注意**：$\lambda_{\text{GC}} > 1$ 也可能来自真实的多基因效应（polygenic signal），此时应使用线性混合模型（LMM），见 Notebook 02。


## 示例 Worked Example

演示两群体在 PC1 上分离，以及加入 PC 前后 $\lambda_{\text{GC}}$ 的变化。

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile, os

rng = np.random.default_rng(2025)
tmpdir = tempfile.mkdtemp()

# ── 合成两群体数据 ──────────────────────────────────────────────
# 群体 A（类欧洲裔）：n_A 个样本
# 群体 B（类非洲裔）：n_B 个样本
n_A, n_B = 300, 200
n = n_A + n_B
m = 2000          # SNP 数（含分层 SNP 和无关 SNP）

# 群体特异性等位基因频率
# 一半 SNP 有群体间频率差异（分层 SNP）
n_strat = m // 2
aaf_A = rng.uniform(0.05, 0.45, m)
aaf_B = aaf_A.copy()
# 给分层 SNP 添加频率差异（Fst ~ 0.15 类似人类群体间）
delta = rng.uniform(0.10, 0.30, n_strat)
aaf_B[:n_strat] = np.clip(aaf_A[:n_strat] + delta, 0.02, 0.98)

G_A = rng.binomial(2, aaf_A, (n_A, m)).astype(float)
G_B = rng.binomial(2, aaf_B, (n_B, m)).astype(float)
G = np.vstack([G_A, G_B])
pop_label = np.array(["A"] * n_A + ["B"] * n_B)

print(f"基因型矩阵 G.shape = {G.shape}  (样本 x SNP)")
print(f"群体 A 样本数 = {n_A},  群体 B 样本数 = {n_B}")
print(f"分层 SNP 数 = {n_strat},  无差异 SNP 数 = {m - n_strat}")


In [ ]:
# ── 实现 genotype_pca（镜像 b8-genopca）──────────────────────
def genotype_pca(G, k):
    """标准化基因型后做 PCA，返回前 k 个主成分 (n, k)，用于群体结构校正。"""
    G = np.asarray(G, dtype=float)
    Gs = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
    return PCA(n_components=k).fit_transform(Gs)

# 提取前 10 个 PC
PCs = genotype_pca(G, k=10)
print(f"PC 矩阵 shape = {PCs.shape}  (n_samples x k_PCs)")

# 方差解释比例（scree plot 数据）
Gs = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
pca_full = PCA(n_components=10).fit(Gs)
var_exp = pca_full.explained_variance_ratio_

print("\n各 PC 解释的方差比例：")
for i, v in enumerate(var_exp, 1):
    print(f"  PC{i}: {v*100:.2f}%")
print(f"  前 2 个 PC 合计: {var_exp[:2].sum()*100:.1f}%")


In [ ]:
# ── 可视化：两群体在 PC1-PC2 上的分离 ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

mask_A = pop_label == "A"
mask_B = pop_label == "B"

ax = axes[0]
ax.scatter(PCs[mask_A, 0], PCs[mask_A, 1], s=12, alpha=0.5,
           color="#2166AC", label="群体 A")
ax.scatter(PCs[mask_B, 0], PCs[mask_B, 1], s=12, alpha=0.5,
           color="#D6604D", label="群体 B")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("基因型 PCA：两群体在 PC1 上分离")
ax.legend(fontsize=9)

# Scree plot
ax2 = axes[1]
ax2.bar(range(1, 11), var_exp * 100, color="steelblue", edgecolor="white")
ax2.set_xlabel("主成分 PC")
ax2.set_ylabel("解释方差 (%)")
ax2.set_title("Scree Plot（碎石图）")
ax2.set_xticks(range(1, 11))

fig.tight_layout()
out = os.path.join(tmpdir, "pca_scatter.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"PCA 散点图已保存：{out}")

# 验证 PC1 在两群体间均值差异显著
t_stat, p_t2 = stats.ttest_ind(PCs[mask_A, 0], PCs[mask_B, 0])
print(f"\nPC1 两群体均值差异 t 检验: t={t_stat:.2f}, p={p_t2:.2e}")
print(f"  群体 A PC1 均值: {PCs[mask_A,0].mean():.4f}")
print(f"  群体 B PC1 均值: {PCs[mask_B,0].mean():.4f}")
print("  (两群体在 PC1 上显著分离)")


In [ ]:
# ── 合成表型（群体间均值不同）────────────────────────────────
# 表型：群体 A 均值 = 0, 群体 B 均值 = 1.5（模拟非遗传的环境差异）
# 这会与分层 SNP 产生虚假关联

pop_offset = np.where(pop_label == "B", 1.5, 0.0)
y = pop_offset + rng.standard_normal(n) * 0.8

print(f"表型 y: 均值={y.mean():.4f}, 标准差={y.std():.4f}")
print(f"  群体 A 均值: {y[mask_A].mean():.4f}")
print(f"  群体 B 均值: {y[mask_B].mean():.4f}")
print("注意：群体间表型均值相差约 1.5 个单位")


In [ ]:
# ── 关联测试：不加 PC（有群体分层）────────────────────────────
def assoc_simple(G, y, covariates=None):
    """向量化关联测试（FWL 残差化）。返回 p 值数组。"""
    G = np.asarray(G, dtype=float)
    y = np.asarray(y, dtype=float)
    n_s = G.shape[0]
    C = np.ones((n_s, 1)) if covariates is None else np.column_stack([np.ones(n_s), covariates])
    Cpinv = np.linalg.pinv(C)
    yr = y - C @ (Cpinv @ y)
    Gr = G - C @ (Cpinv @ G)
    dof = n_s - C.shape[1] - 1
    Sxx = (Gr ** 2).sum(axis=0)
    beta = (Gr * yr[:, None]).sum(axis=0) / Sxx
    resid_sq = ((yr[:, None] - Gr * beta[None, :]) ** 2).sum(axis=0)
    sigma2 = resid_sq / dof
    se = np.sqrt(sigma2 / Sxx)
    t = beta / se
    p = 2 * stats.t.sf(np.abs(t), dof)
    return t, p

def genomic_lambda(t_vals):
    """lambda_GC = median(chi2_obs) / 0.4549."""
    chi2_obs = t_vals ** 2
    return float(np.median(chi2_obs) / stats.chi2.ppf(0.5, df=1))

# 不加 PC：群体分层膨胀
t_no_pc, p_no_pc = assoc_simple(G, y, covariates=None)
lam_no_pc = genomic_lambda(t_no_pc)

# 加入前 10 个 PC：校正群体分层
t_with_pc, p_with_pc = assoc_simple(G, y, covariates=PCs[:, :10])
lam_with_pc = genomic_lambda(t_with_pc)

print(f"lambda_GC（不加 PC）  = {lam_no_pc:.4f}  (群体分层导致膨胀)")
print(f"lambda_GC（加入 PC）  = {lam_with_pc:.4f}  (校正后接近 1.0)")
print(f"膨胀改善量: {lam_no_pc - lam_with_pc:.4f}")


In [ ]:
# ── QQ 图：校正前后对比 ────────────────────────────────────────
def qq_points(pvals):
    p = np.sort(np.asarray(pvals, dtype=float))
    n_p = len(p)
    expected = -np.log10(np.arange(1, n_p + 1) / (n_p + 1))
    observed = -np.log10(np.clip(p, 1e-300, 1.0))
    return expected[::-1], observed[::-1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, (t_v, p_v, lam, title, color) in zip(axes, [
    (t_no_pc,   p_no_pc,   lam_no_pc,   rf"不加 PC ($\lambda_{{GC}}$={lam_no_pc:.3f})",   "tomato"),
    (t_with_pc, p_with_pc, lam_with_pc, rf"加入 PC ($\lambda_{{GC}}$={lam_with_pc:.3f})", "steelblue"),
]):
    exp_lp, obs_lp = qq_points(p_v)
    step = max(1, len(exp_lp) // 400)
    ax.scatter(exp_lp[::step], obs_lp[::step], s=8, alpha=0.6, color=color)
    lim = max(exp_lp.max(), obs_lp.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", linewidth=0.8, label="y=x")
    ax.set_xlim(0, exp_lp.max() * 1.05)
    ax.set_ylim(0, obs_lp.max() * 1.05)
    ax.set_xlabel(r"期望 $-\log_{10}(p)$")
    ax.set_ylabel(r"观测 $-\log_{10}(p)$")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle("QQ 图：群体分层校正前后对比", fontsize=11)
fig.tight_layout()
out = os.path.join(tmpdir, "qq_stratification.png")
fig.savefig(out, dpi=120)
plt.close(fig)
print(f"QQ 图已保存：{out}")
print()
print("=== 数值验证 ===")
print(f"不加 PC - 超过 1e-5 的虚假 SNP 数: {(p_no_pc < 1e-5).sum()}")
print(f"加 PC   - 超过 1e-5 的 SNP 数:     {(p_with_pc < 1e-5).sum()}")
print("加入 PC 后虚假信号大幅减少（无真实 causal SNP）")


In [ ]:
# ── 验证 genotype_pca 的标准化逻辑 ──────────────────────────
print("=== genotype_pca 验证 ===")

# 与 sklearn PCA 直接对比
Gs_ref = (G - G.mean(axis=0)) / (G.std(axis=0) + 1e-8)
pca_ref = PCA(n_components=5).fit_transform(Gs_ref)
pcs_ours = genotype_pca(G, k=5)

# PC 方向可能相差符号，取绝对值对比相关性
for i in range(5):
    corr = np.corrcoef(pca_ref[:, i], pcs_ours[:, i])[0, 1]
    print(f"  PC{i+1} 相关系数 |r| = {abs(corr):.8f}  (应接近 1.0)")

print()
print("PC 均值（应接近 0）:", np.abs(pcs_ours.mean(axis=0)).max().round(10))
print("genotype_pca 验证通过")


## 动手 Exercise

以下合成数据 `G_ex`、`y_ex` 包含三个群体（pop 0/1/2），且有真实的 causal SNP（SNP 0 有效应 0.5）。请你：

1. 调用 `genotype_pca(G_ex, k=5)` 提取前 5 个 PC，并绘制 PC1 vs PC2 的散点图（用 pop 标签着色）。
2. 对无协变量和有前 5 个 PC 两种情况分别运行 `assoc_simple`，打印各自的 $\lambda_{\text{GC}}$。
3. 验证加入 PC 后 $\lambda_{\text{GC}}$ 更接近 1.0。

```python
rng_ex = np.random.default_rng(31415)
n_pop = 150
m_ex  = 500
# 三个群体等位基因频率
aaf0 = rng_ex.uniform(0.05, 0.40, m_ex)
aaf1 = np.clip(aaf0 + rng_ex.uniform(0.10, 0.25, m_ex), 0.02, 0.98)
aaf2 = np.clip(aaf0 - rng_ex.uniform(0.05, 0.20, m_ex), 0.02, 0.98)
G_ex = np.vstack([
    rng_ex.binomial(2, aaf0, (n_pop, m_ex)),
    rng_ex.binomial(2, aaf1, (n_pop, m_ex)),
    rng_ex.binomial(2, aaf2, (n_pop, m_ex)),
]).astype(float)
pop_ex = np.array([0]*n_pop + [1]*n_pop + [2]*n_pop)
pop_offset_ex = np.where(pop_ex == 1, 2.0, np.where(pop_ex == 2, -1.5, 0.0))
y_ex = G_ex[:, 0] * 0.5 + pop_offset_ex + rng_ex.standard_normal(3 * n_pop)

# 你的代码写在这里
```

> **提示**：PC1 和 PC2 通常能区分三个群体。不加 PC 时 $\lambda_{\text{GC}}$ 应明显 > 1。


## 延伸阅读 Further Reading

1. **Price et al. (2006)**. "Principal components analysis corrects for stratification in genome-wide association studies." *Nature Genetics.* PCA 校正群体分层的奠基论文。
2. **Patterson, Price & Reich (2006)**. "Population structure and eigenanalysis." *PLOS Genetics.* 遗传 PCA 理论基础（Tracy-Widom 检验）。
3. **Devlin & Roeder (1999)**. "Genomic control for association studies." *Biometrics.* $\lambda_{\text{GC}}$ 原始文献。
4. **Novembre et al. (2008)**. "Genes mirror geography within Europe." *Nature.* 直观演示欧洲人群 PC1/PC2 与地理的对应。


## 关联练习 Related Assignment

完成 **b8-genopca** 编程作业：在 `progress/<你>/work/b8-genopca/solution.py` 中实现
`genotype_pca(G, k)` 函数（标准化 + PCA，返回 `(n, k)` 主成分矩阵）。

评测命令：

```bash
python check.py 01-genopca
```

> 函数签名与本课示例完全一致，直接参照上方 `genotype_pca` 实现即可。
